In [0]:
%run ../setup/config

In [0]:
%run ../setup/utils

In [0]:
movies_metadata_df = spark.read.format("delta").load(f"{silver_folder_path}/movies_metadata")
ratings_df = spark.read.format("delta").load(f"{silver_folder_path}/ratings")
links_df = spark.read.format("delta").load(f"{silver_folder_path}/links")
crew_df = spark.read.format("delta").load(f"{silver_folder_path}/crew")

movies_metadata_df.printSchema()
ratings_df.printSchema()
links_df.printSchema()
crew_df.printSchema()


In [0]:
from pyspark.sql import functions as F

final_movies_df = (
  ratings_df
    .join(links_df, links_df.movie_id == ratings_df.movie_id, "inner")
    .join(movies_metadata_df, links_df.tmbd_id == movies_metadata_df.id, "inner")
    .join(crew_df, movies_metadata_df.id == crew_df.id, "inner")
    .groupBy(movies_metadata_df.id, "title", "crew_id", "crew_job", "crew_name")
    .agg(
        F.avg("rating").alias("average_rating"),
        F.count("user_id").alias("number_of_ratings"),
    )
    .filter("number_of_ratings > 200")
    .filter("crew_job == 'Director'")
    .orderBy(
      F.col("average_rating").desc(),
      F.col("id")
      )
)
display(final_movies_df)

In [0]:
from pyspark.sql.window import Window

df = (
  final_movies_df
    .groupBy("crew_id", "crew_name")
    .agg(
      F.countDistinct("id").alias("movies_directed"),
      F.avg("average_rating").alias("avg_movies_rating"),
    )
    .filter("movies_directed >= 6")
    .withColumn(
      "rank",
      F.rank().over(Window.orderBy(F.col("avg_movies_rating").desc())),
    )
    .orderBy("rank")
)

display(df)